# Mitterrand 1981 — Agent conversationnel

Point d'entrée unique du projet : observation du modèle, baseline sans entraînement, QLoRA sur `google/gemma-4-12B-it`, évaluation avant/après et interface Gradio.

Deux parcours, depuis un environnement neuf :
- **Entraîner et discuter :** mettre `RUN_TRAINING = True` et exécuter les cellules dans l'ordre.
- **Discuter avec l’adaptateur sur Drive :** garder `RUN_TRAINING = False` et `ADAPTER_PATH = "/content/drive/MyDrive/mitterrand-qlora"`, puis exécuter les mêmes cellules. Les cellules d'entraînement sont ignorées.

Dans les deux cas, le modèle de base est chargé une seule fois (section 4), observé (section 5) et testé sans entraînement (section 6). Il reçoit ensuite l'adaptateur LoRA, entraîné (sections 7 à 13) ou rechargé (section 14). La section 15 compare la baseline et le modèle adapté, la section 16 lance l'interface.

**Environnement :** Colab avec GPU L4 ou A100 pour l'entraînement, accès Hugging Face au modèle et jeton `HF_TOKEN` (ou connexion interactive). L'adaptateur peut être sauvegardé sur Drive ou publié sur Hugging Face. Un T4 suffit en mémoire pour discuter, mais le float16 peut être instable avec Gemma.

**Interface :** ouvrir Colab dans Chrome ou Firefox. L'interface s'affiche sous la dernière cellule, sans lien public. L'extension Colab de VS Code ne transfère pas les ports Gradio : ouvrir le notebook dans Colab pour la démonstration.

**Résultats :** la baseline et l'évaluation sont enregistrées dans `RESULTS_DIR` (`Mon Drive/mitterrand-eval` par défaut). L'évaluation crée une grille `grille.csv` à noter à la main, puis ses cellules de synthèse sont relancées. Ce notebook doit être exécuté avant la remise.

## 1. Installation


In [ ]:
%pip install -q -U transformers peft trl bitsandbytes accelerate datasets ipywidgets matplotlib "gradio>=6,<7"


## 2. Configuration

Les hyperparamètres sont prévus pour ~550 exemples : 3 époques, soit environ 200 pas d'optimisation avec un batch effectif de 8. Le meilleur checkpoint (plus faible perte de validation) est rechargé à la fin, ce qui protège du sur-apprentissage.

In [ ]:
RUN_TRAINING = False  # @param {type:"boolean"}
ADAPTER_PATH = "/content/drive/MyDrive/mitterrand-qlora"  # @param {type:"string"}

MODEL_ID = "google/gemma-4-12B-it"  # @param {type:"string"}
OUTPUT_DIR = "gemma-4-12b-mitterrand-qlora"  # @param {type:"string"}

MAX_LENGTH = 1024  # @param {type:"integer"}  les exemples font au plus ~800 tokens
NUM_EPOCHS = 3  # @param {type:"integer"}
LEARNING_RATE = 2e-4  # @param {type:"number"}
EFFECTIVE_BATCH_SIZE = 8  # @param {type:"integer"}
LORA_R = 16  # @param {type:"integer"}
LORA_ALPHA = 32  # @param {type:"integer"}
LORA_DROPOUT = 0.05  # @param {type:"number"}

SAVE_TO_DRIVE = True  # @param {type:"boolean"}
DRIVE_DIR = "/content/drive/MyDrive/mitterrand-qlora"  # @param {type:"string"}

RESULTS_DIR = "/content/drive/MyDrive/mitterrand-eval"  # @param {type:"string"}  baseline et évaluation

### Fichiers du projet

En local, lancer le notebook depuis le dépôt : les données, le prompt et l'app sont utilisés directement. Dans Colab, cette cellule récupère le dépôt. `REPO_BRANCH` doit désigner la branche contenant les fichiers du projet.

Les résultats de la baseline et de l'évaluation vont dans `RESULTS_DIR` : sur Drive dans Colab, pour qu'ils survivent à la fin de la session, et dans `evaluation/` du dépôt hors de Colab.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Kodaiii/nlp-mitterand.git"  # @param {type:"string"}
REPO_BRANCH = "dataset-qlora-mitterrand"  # @param {type:"string"}  mettre "main" une fois la PR fusionnée

_local_roots = [Path.cwd(), *Path.cwd().parents]
REPO_DIR = next((p for p in _local_roots if (p / "app" / "app.py").is_file() and (p / "data" / "system_prompt.txt").is_file()), None)
if REPO_DIR is None:
    REPO_DIR = Path("/content/nlp-mitterand")
    if not (REPO_DIR / "app" / "app.py").is_file():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
sys.path.insert(0, str(REPO_DIR))
print(f"Dépôt : {REPO_DIR}")

RESULTS_DIR = Path(RESULTS_DIR)
if str(RESULTS_DIR).startswith("/content/drive"):
    try:
        from google.colab import drive

        drive.mount("/content/drive")
    except ImportError:  # hors de Colab : les résultats restent dans le dépôt
        RESULTS_DIR = REPO_DIR / "evaluation"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Résultats : {RESULTS_DIR}")

## 3. Connexion à Hugging Face

In [ ]:
from huggingface_hub import login

try:
    from google.colab import userdata

    login(token=userdata.get("HF_TOKEN"))
    print("Connecté avec le secret HF_TOKEN.")
except Exception as error:  # secret absent, ou extension VS Code (userdata n'y est pas disponible)
    print(f"Secret HF_TOKEN indisponible ({error}), connexion interactive :")
    from huggingface_hub import notebook_login

    notebook_login()

## 4. Chargement du modèle de base en 4 bits

Le modèle est chargé une seule fois, pour les deux parcours. Il sert tel quel à l'observation (section 5) et à la baseline (section 6), puis reçoit l'adaptateur LoRA : entraîné en sections 7 à 13, ou rechargé depuis Drive en section 14.

Quantification NF4 avec double quantification, calcul en bfloat16 (configuration du guide QLoRA officiel de Gemma 4). Gemma 4 se charge avec `AutoModelForMultimodalLM` parce qu'il accepte aussi des images ; seule sa partie texte, un décodeur causal, sert ici.

In [ ]:
import torch
from transformers import AutoModelForMultimodalLM, AutoProcessor, BitsAndBytesConfig

gpu = torch.cuda.get_device_properties(0)
GPU_MEMORY_GB = gpu.total_memory / 1024**3
print(f"GPU : {gpu.name} ({GPU_MEMORY_GB:.0f} Go)")

if torch.cuda.is_bf16_supported():
    torch_dtype = torch.bfloat16
else:
    torch_dtype = torch.float16
    print("⚠️  Pas de bfloat16 sur ce GPU : le float16 peut être instable avec Gemma. Préférez un L4 ou un A100.")

model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    dtype=torch_dtype,
    device_map="auto",
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch_dtype,
        bnb_4bit_quant_storage=torch_dtype,
    ),
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
tokenizer = processor.tokenizer

# Prompt système du projet, identique à celui des données d'entraînement.
PROJECT_SYSTEM_PROMPT = (REPO_DIR / "data" / "system_prompt.txt").read_text(encoding="utf-8").strip()
SYSTEM_PROMPT = PROJECT_SYSTEM_PROMPT

print(f"Mémoire GPU utilisée : {torch.cuda.memory_allocated() / 1024**3:.1f} Go")

## 5. Observer le modèle : tokenisation, logits et génération

On observe le modèle de base sur une phrase courte, sans chat template, en suivant le trajet d'un texte dans un Transformer causal (TP Transformers) :

1. le tokenizer découpe le texte en sous-mots et renvoie leurs identifiants ;
2. la matrice d'embeddings associe un vecteur à chaque identifiant ;
3. les couches d'attention causale transforment ces vecteurs : chaque position ne voit que les tokens qui la précèdent ;
4. l'unembedding (`lm_head`) projette l'état caché de chaque position sur tout le vocabulaire : ce sont les logits ;
5. la softmax transforme les logits de la dernière position en distribution du prochain token. On y choisit un token (argmax ou échantillonnage), on l'ajoute au texte et on recommence : c'est la génération autorégressive.

In [ ]:
OBSERVATION_TEXT = "Le 10 mai 1981, François Mitterrand a été élu président de la"

encoded = tokenizer(OBSERVATION_TEXT, return_tensors="pt")
token_ids = encoded["input_ids"][0].tolist()
print(f"{len(OBSERVATION_TEXT)} caractères → {len(token_ids)} tokens\n")
print("position  identifiant  token")
for position, token_id in enumerate(token_ids):
    print(f"{position:>8}  {token_id:>11}  {tokenizer.convert_ids_to_tokens(token_id)!r}")

Chaque identifiant est un numéro de ligne dans la matrice d'embeddings. Les tokens spéciaux éventuels, comme le début de séquence, apparaissent en tête. Les mots fréquents tiennent souvent en un seul token ; les mots rares et les nombres sont découpés en plusieurs sous-mots.

In [ ]:
inputs = {key: value.to(model.device) for key, value in encoded.items()}
with torch.no_grad():
    outputs = model(**inputs, output_hidden_states=True)

logits = outputs.logits
last_hidden = outputs.hidden_states[-1]
embedding_matrix = model.get_input_embeddings().weight
lm_head = model.get_output_embeddings()

print(f"Embeddings          : {tuple(embedding_matrix.shape)}  (taille du vocabulaire, dimension du modèle)")
print(f"Dernier état caché  : {tuple(last_hidden.shape)}  (lot, positions, dimension du modèle)")
print(f"Unembedding lm_head : {lm_head.in_features} → {lm_head.out_features}  (dimension du modèle → vocabulaire)")
print(f"Logits              : {tuple(logits.shape)}  (lot, positions, taille du vocabulaire)")

# Unembedding : les logits de la dernière position sont la projection de son état caché par lm_head.
recomputed = lm_head(last_hidden[:, -1]).float()
softcap = getattr(model.config.get_text_config(), "final_logit_softcapping", None)
if softcap:  # Gemma peut plafonner les logits en douceur : softcap × tanh(logit / softcap)
    recomputed = softcap * torch.tanh(recomputed / softcap)
gap = (recomputed - logits[:, -1].float()).abs().max().item()
same = recomputed[0].argmax().item() == logits[0, -1].argmax().item()
print(f"\nlm_head(dernier état caché) comparé aux logits : écart max. {gap:.4f}, même token le plus probable : {same}")

# Attention causale : ce qui est prédit à une position ne dépend pas des tokens qui la suivent.
cut = len(token_ids) // 2
with torch.no_grad():
    prefix_logits = model(**{key: value[:, :cut] for key, value in inputs.items()}).logits
gap = (prefix_logits[0, -1].float() - logits[0, cut - 1].float()).abs().max().item()
same = prefix_logits[0, -1].argmax().item() == logits[0, cut - 1].argmax().item()
print(f"Position {cut - 1}, calculée sur les {cut} premiers tokens puis sur la phrase entière : écart max. {gap:.4f}, même token prédit : {same}")

Les logits ont une ligne par position : à chaque position, le modèle prédit le token suivant, et seule la dernière ligne sert à la génération. La première vérification relie les logits à l'unembedding. La seconde illustre le masque causal : couper la phrase après une position ne change pas ce qui y est prédit, aux arrondis de calcul près.

In [ ]:
import matplotlib.pyplot as plt

next_token_logits = logits[0, -1].float()


def entropy(probabilities):
    return -(probabilities * torch.log(probabilities.clamp_min(1e-12))).sum().item()


probs = torch.softmax(next_token_logits, dim=-1)
print(f"Softmax sur {probs.numel()} tokens : somme des probabilités {probs.sum().item():.4f}, entropie {entropy(probs):.2f} nats\n")
print("rang  probabilité   logit  token")
top = torch.topk(probs, 10)
for rank, (probability, token_id) in enumerate(zip(top.values.tolist(), top.indices.tolist()), 1):
    print(f"{rank:>4}  {probability:>11.3f}  {next_token_logits[token_id].item():>6.1f}  {tokenizer.decode([token_id])!r}")

print("\nEffet de la température T (softmax des logits divisés par T) :")
for temperature in (0.3, 0.7, 1.0, 1.5):
    sorted_probs = torch.softmax(next_token_logits / temperature, dim=-1).sort(descending=True).values
    kept_by_top_p = int((sorted_probs.cumsum(0) < 0.95).sum().item()) + 1
    print(
        f"T = {temperature:<3}  p(1er token) {sorted_probs[0].item():.3f}  entropie {entropy(sorted_probs):.2f} nats  "
        f"masse des 50 premiers (top-k 50) {sorted_probs[:50].sum().item():.3f}  tokens gardés par top-p 0,95 : {kept_by_top_p}"
    )

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8), sharey=True)
labels = [repr(tokenizer.decode([token_id])) for token_id in top.indices.tolist()]
for axis, temperature in zip(axes, (0.3, 1.0, 1.5)):
    values = torch.softmax(next_token_logits / temperature, dim=-1)[top.indices].tolist()
    axis.bar(range(len(values)), values, color="tab:blue")
    axis.set_xticks(range(len(values)), labels, rotation=60, fontsize=8)
    axis.set_title(f"T = {temperature}")
    axis.grid(axis="y", alpha=0.3)
axes[0].set_ylabel("probabilité")
fig.suptitle("Les 10 prochains tokens les plus probables selon la température")
plt.tight_layout()
plt.show()

La température divise les logits avant la softmax. En dessous de 1, elle accentue les écarts : le premier token prend l'essentiel de la masse et la génération se rapproche de l'argmax. Au-dessus de 1, elle aplatit la distribution et laisse leur chance à des tokens rares. Top-k garde toujours k tokens, quelle que soit la forme de la distribution. Top-p garde le plus petit ensemble qui couvre la probabilité p : il est petit quand le modèle est sûr de lui, grand quand il hésite.

In [ ]:
from transformers import set_seed

STRATEGIES = {
    "argmax": {"do_sample": False},
    "T = 0,3": {"do_sample": True, "temperature": 0.3, "top_k": 0, "top_p": 1.0},
    "T = 1,0": {"do_sample": True, "temperature": 1.0, "top_k": 0, "top_p": 1.0},
    "T = 1,5": {"do_sample": True, "temperature": 1.5, "top_k": 0, "top_p": 1.0},
    "top-k 50, T = 1,0": {"do_sample": True, "temperature": 1.0, "top_k": 50, "top_p": 1.0},
    "top-p 0,95, T = 1,0": {"do_sample": True, "temperature": 1.0, "top_k": 0, "top_p": 0.95},
}

print(f"{OBSERVATION_TEXT} …\n")
for name, strategy in STRATEGIES.items():
    for seed in (0, 1):
        set_seed(seed)
        with torch.no_grad():
            output = model.generate(**inputs, max_new_tokens=25, **strategy)
        continuation = tokenizer.decode(output[0, inputs["input_ids"].shape[1] :], skip_special_tokens=True)
        print(f"{name:<20} graine {seed} : …{' '.join(continuation.split())}")
    print()

L'argmax donne la même suite quelle que soit la graine : la génération est déterministe, mais elle prend toujours le token le plus probable, ce qui favorise les formules convenues et les répétitions. L'échantillonnage varie d'une graine à l'autre, d'autant plus que la température est élevée ; sans filtre, une température haute laisse passer des tokens très improbables. Top-k et top-p coupent cette queue de distribution. Ces observations guident les paramètres de la baseline.

## 6. Baseline sans entraînement

Première version de l'agent, sans aucun entraînement : le modèle de base, un message système, le format de conversation attendu par le tokenizer (`apply_chat_template`) et un historique de taille contrôlée.

**Paramètres de génération**, d'après la section 5 :
- **échantillonnage plutôt qu'argmax** : l'argmax est déterministe et répète volontiers les mêmes formules, alors qu'on attend des réponses variées ;
- **température 0,7** : sous 1, elle concentre la distribution sur les tokens plausibles sans figer les tournures ;
- **top-p 0,95** : il écarte la queue de la distribution, d'où viennent les mots incongrus, tout en gardant plus de candidats quand le modèle hésite ;
- **400 nouveaux tokens au plus** : la longueur des réponses du jeu d'entraînement est mesurée plus bas ;
- **historique limité aux 5 derniers échanges** (10 messages), comme dans l'interface : le prompt reste court et le temps de génération borné.

La même fonction `ask()` sert à la baseline, au modèle adapté et à Gradio.

In [ ]:
from transformers import set_seed

MAX_HISTORY_TURNS = 5  # comme app/app.py : seuls les 5 derniers échanges sont renvoyés au modèle
GENERATION_KWARGS = {"do_sample": True, "temperature": 0.7, "top_p": 0.95}


def build_messages(question, history=None, system_prompt=None, shots=()):
    # shots : échanges d'exemple placés avant la conversation (variante few-shot)
    history = list(history or [])[-2 * MAX_HISTORY_TURNS :]
    return [{"role": "system", "content": system_prompt or SYSTEM_PROMPT}, *shots, *history, {"role": "user", "content": question}]


def ask(question, history=None, max_new_tokens=400, system_prompt=None, shots=()):
    messages = build_messages(question, history, system_prompt, shots)
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, **GENERATION_KWARGS)
    return processor.decode(output[0][inputs["input_ids"].shape[-1] :], skip_special_tokens=True).strip()


# Le format de conversation attendu par le tokenizer, tel que le modèle le reçoit :
print(processor.apply_chat_template(build_messages("Bonjour, Monsieur Mitterrand."), tokenize=False, add_generation_prompt=True))

In [ ]:
import json


def read_jsonl(name):
    with open(REPO_DIR / "data" / name, encoding="utf-8") as f:
        return [json.loads(line) for line in f]


train_examples = read_jsonl("mitterrand_train.jsonl")
val_examples = read_jsonl("mitterrand_val.jsonl")

answer_lengths = sorted(
    len(tokenizer(message["content"], add_special_tokens=False)["input_ids"])
    for example in train_examples
    for message in example["messages"]
    if message["role"] == "assistant"
)
print(
    f"Réponses du jeu d'entraînement ({len(answer_lengths)}) : médiane {answer_lengths[len(answer_lengths) // 2]} tokens, "
    f"95e centile {answer_lengths[int(0.95 * len(answer_lengths))]}, maximum {answer_lengths[-1]}"
)
print(f"Réponses de plus de 400 tokens : {sum(length > 400 for length in answer_lengths)}")

### Comparaison de trois configurations de prompt

- **A. Prompt court, zero-shot** : une phrase, « Tu es François Mitterrand en 1981. Réponds en français. »
- **B. Prompt détaillé, zero-shot** : le prompt du projet (`data/system_prompt.txt`, 181 mots), qui fixe l'époque, le style, le vouvoiement et la conduite hors périmètre.
- **C. Prompt détaillé, few-shot** : B suivi de deux échanges du jeu d'entraînement, une question sur le programme et une question hors époque.

Deux mesures, sur les données de validation, qui n'ont jamais servi à mettre à jour les poids :
1. **la perte sur les réponses de référence** : perte moyenne par token de la réponse attendue, sachant le prompt et la question, sur les 58 exemples de validation. Plus elle est basse, plus la configuration rapproche le modèle des réponses visées ;
2. **la lecture des réponses générées** pour 5 questions de validation, avec deux indicateurs automatiques de format : listes ou markdown, et tutoiement.

La configuration de plus faible perte devient la baseline ; ses réponses servent de référence.

In [ ]:
import math

SHORT_SYSTEM_PROMPT = "Tu es François Mitterrand en 1981. Réponds en français."

# Deux échanges du jeu d'entraînement pour la variante few-shot : une proposition du programme, une question hors époque.
FEW_SHOT_QUESTIONS = [
    "Monsieur Mitterrand, quelle est votre position sur l'intervention soviétique en Afghanistan ?",
    "Que pensez-vous d'Emmanuel Macron ?",
]
FEW_SHOT = [
    message
    for question in FEW_SHOT_QUESTIONS
    for message in next(example["messages"][1:3] for example in train_examples if example["messages"][1]["content"] == question)
]

PROMPT_CONFIGS = {
    "A. prompt court, zero-shot": {"system_prompt": SHORT_SYSTEM_PROMPT, "shots": []},
    "B. prompt détaillé, zero-shot": {"system_prompt": PROJECT_SYSTEM_PROMPT, "shots": []},
    "C. prompt détaillé, few-shot": {"system_prompt": PROJECT_SYSTEM_PROMPT, "shots": FEW_SHOT},
}


def split_example(example):
    # (historique sans le prompt système, dernière question, réponse de référence)
    messages = example["messages"]
    return messages[1:-2], messages[-2]["content"], messages[-1]["content"]


def reference_loss(messages, reference):
    # Perte moyenne par token sur la réponse de référence, sachant la conversation qui la précède.
    prompt = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt"
    )
    prompt_length = prompt["input_ids"].shape[1]
    answer_ids = processor.tokenizer(reference + "<turn|>", add_special_tokens=False, return_tensors="pt")["input_ids"]
    inputs = {}
    for key, value in prompt.items():
        if key == "input_ids":
            inputs[key] = torch.cat([value, answer_ids], dim=1)
        elif torch.is_tensor(value) and value.dim() == 2 and value.shape[1] == prompt_length:
            filler = torch.ones_like(answer_ids) if key == "attention_mask" else torch.zeros_like(answer_ids)
            inputs[key] = torch.cat([value, filler.to(value.dtype)], dim=1)
    labels = inputs["input_ids"].clone()
    labels[:, :prompt_length] = -100  # seule la réponse compte, fermeture <turn|> comprise
    with torch.no_grad():
        output = model(**{key: value.to(model.device) for key, value in inputs.items()}, labels=labels.to(model.device))
    return output.loss.item()


prompt_losses = {}
for name, config in PROMPT_CONFIGS.items():
    losses = [
        reference_loss(build_messages(question, history, **config), reference)
        for history, question, reference in map(split_example, val_examples)
    ]
    prompt_losses[name] = sum(losses) / len(losses)
    print(f"{name:<30} perte {prompt_losses[name]:.3f}  (perplexité {math.exp(prompt_losses[name]):.1f})")

In [ ]:
import random
import re

import pandas as pd
from IPython.display import display

PROMPT_SEED = 42


def format_indicators(answer):
    return {
        "markdown": bool(re.search(r"(^|\n)\s*([-*•#]|\d+[.)])\s|\*\*", answer)),
        "tutoiement": bool(re.search(r"\b(tu|toi)\b", answer, re.IGNORECASE)),
        "mots": len(answer.split()),
    }


dev_examples = random.Random(0).sample(val_examples, 5)
prompt_answers = {name: [] for name in PROMPT_CONFIGS}
for history, question, reference in map(split_example, dev_examples):
    print(f"❓ {question}\n")
    for name, config in PROMPT_CONFIGS.items():
        set_seed(PROMPT_SEED)
        answer = ask(question, history, **config)
        prompt_answers[name].append(answer)
        print(f"[{name}]\n{answer}\n")
    print(f"[référence]\n{reference}\n\n{'─' * 80}\n")

rows = []
for name, answers in prompt_answers.items():
    indicators = [format_indicators(answer) for answer in answers]
    rows.append({
        "configuration": name,
        "perte (validation)": round(prompt_losses[name], 3),
        "perplexité": round(math.exp(prompt_losses[name]), 1),
        "listes ou markdown": f"{sum(i['markdown'] for i in indicators)}/{len(indicators)}",
        "tutoiement": f"{sum(i['tutoiement'] for i in indicators)}/{len(indicators)}",
        "longueur moyenne (mots)": round(sum(i["mots"] for i in indicators) / len(indicators)),
    })
display(pd.DataFrame(rows).set_index("configuration"))

BEST_PROMPT_CONFIG = min(prompt_losses, key=prompt_losses.get)
BASELINE_CONFIG = PROMPT_CONFIGS[BEST_PROMPT_CONFIG]
print(f"Configuration retenue pour la baseline : {BEST_PROMPT_CONFIG}")

### Réponses de référence de la baseline

Ces cinq questions ne figurent pas dans les données. Elles sont reposées au modèle adapté en section 13 et font partie de l'évaluation de la section 15.

In [ ]:
TEST_QUESTIONS = [
    "Que proposez-vous pour les retraites ?",
    "Monsieur Mitterrand, pourquoi voudriez-vous abolir la peine de mort alors que les Français y sont favorables ?",
    "tu penses quoi de TikTok ?",
    "Pourquoi voter pour vous plutôt que pour Giscard ?",
    "Que pensez-vous de la force de dissuasion nucléaire ?",
]

baseline_answers = {}
for question in TEST_QUESTIONS:
    set_seed(PROMPT_SEED)
    baseline_answers[question] = ask(question, **BASELINE_CONFIG)
    print(f"❓ {question}\n\n{baseline_answers[question]}\n\n{'─' * 80}\n")

baseline_file = RESULTS_DIR / "baseline_reference.json"
baseline_file.write_text(
    json.dumps({"configuration": BEST_PROMPT_CONFIG, "reponses": baseline_answers}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(f"Réponses de référence enregistrées : {baseline_file}")

## 7. Données

Chaque exemple est au format `{"messages": [system, user, assistant, ...]}`. Le prompt système est le même partout ; on le récupère dans les données pour le réutiliser à l'inférence.

In [ ]:
if RUN_TRAINING:
    import os

    import ipywidgets as widgets
    from IPython.display import display

    DATA_FILES = {split: str(REPO_DIR / "data" / name) for split, name in {"train": "mitterrand_train.jsonl", "validation": "mitterrand_val.jsonl"}.items()}
    missing = [name for name in DATA_FILES.values() if not os.path.exists(name)]

    # Widget de téléversement : il marche dans Colab comme dans VS Code (files.upload() n'existe pas dans l'extension).
    uploader = widgets.FileUpload(accept=".jsonl", multiple=True, description="Fichiers .jsonl")
    if missing:
        print(f"Fichiers manquants : {missing}.\nSélectionnez-les avec le bouton ci-dessous, puis exécutez la cellule suivante.")
        display(uploader)
    else:
        print("Fichiers de données déjà présents.")

In [ ]:
if RUN_TRAINING:
    def uploaded_files(value):
        # ipywidgets 7 (Colab) renvoie {nom: {"content": ...}}, ipywidgets 8 un tuple de dicts avec une clé "name".
        if isinstance(value, dict):
            return [(name, item["content"]) for name, item in value.items()]
        return [(item["name"], item["content"]) for item in value]


    for name, content in uploaded_files(uploader.value):
        with open(REPO_DIR / "data" / Path(name).name, "wb") as f:
            f.write(bytes(content))
        print(f"Enregistré : {name}")

    missing = [name for name in DATA_FILES.values() if not os.path.exists(name)]
    assert not missing, f"Toujours manquants : {missing}. Téléversez-les avec le bouton de la cellule précédente."

    from datasets import load_dataset

    dataset = load_dataset("json", data_files=DATA_FILES)
    print(dataset)

    SYSTEM_PROMPT = dataset["train"][0]["messages"][0]["content"]
    assert dataset["train"][0]["messages"][0]["role"] == "system", "Les données doivent commencer par un message system."
    print("\nPrompt système :\n" + SYSTEM_PROMPT)

## 8. Chat template et longueur des exemples

Le template de Gemma 4 découpe la conversation en tours `<|turn>system`, `<|turn>user` et `<|turn>model`, chacun fermé par `<turn|>`.

Sur le 12B (comme sur le 26B et le 31B), quand le mode « thinking » est désactivé, le template ouvre chaque réponse par un **canal de pensée vide** (`<|channel>thought\n<channel|>`). Google recommande de reproduire ce canal à l'entraînement quand les données n'ont pas de raisonnement explicite. On détecte donc ce que le template ajoute au moment de générer, et on l'insère au début de chaque tour du modèle dans les exemples d'entraînement. Ainsi, entraînement et inférence voient exactement le même format.

In [ ]:
if RUN_TRAINING:
    import re

    tokenizer = processor.tokenizer
    tokenizer.padding_side = "right"  # padding à droite pour l'entraînement
    PROCESSOR_ADDS_BOS = processor(text="test", return_tensors="pt")["input_ids"][0][0].item() == tokenizer.bos_token_id
    MODEL_TURN_OPEN = "<|turn>model\n"

    # Ce que le template place après « <|turn>model\n » quand on lui demande de générer.
    _probe = processor.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": "Bonjour"}],
        tokenize=False,
        add_generation_prompt=True,
    )
    assert MODEL_TURN_OPEN in _probe, f"Format inattendu, vérifiez le template :\n{_probe}"
    GENERATION_PREFIX = _probe[_probe.rindex(MODEL_TURN_OPEN) + len(MODEL_TURN_OPEN) :]
    print(f"Préfixe de génération après {MODEL_TURN_OPEN!r} : {GENERATION_PREFIX!r}")


    def render(messages):
        # Texte au format Gemma 4, avec le même préfixe de génération au début de chaque tour du modèle.
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        if GENERATION_PREFIX:
            not_followed_by_prefix = re.escape(MODEL_TURN_OPEN) + f"(?!{re.escape(GENERATION_PREFIX)})"
            text = re.sub(not_followed_by_prefix, lambda _: MODEL_TURN_OPEN + GENERATION_PREFIX, text)
        text = text.strip()
        # On retire le <bos> du template si le processor en ajoute déjà un, pour éviter de le doubler.
        if PROCESSOR_ADDS_BOS and tokenizer.bos_token and text.startswith(tokenizer.bos_token):
            text = text[len(tokenizer.bos_token) :]
        return text


    print(render(dataset["train"][0]["messages"]))

    lengths = [len(processor(text=render(ex["messages"]))["input_ids"][0]) for split in dataset for ex in dataset[split]]
    too_long = sum(n > MAX_LENGTH for n in lengths)
    print(f"\nTokens par exemple : max {max(lengths)}, moyenne {sum(lengths) / len(lengths):.0f}")
    print(f"Exemples tronqués à {MAX_LENGTH} tokens : {too_long}" + ("  ⚠️ augmentez MAX_LENGTH" if too_long else ""))

## 9. Masquage de la perte : n'apprendre que les réponses de Mitterrand

La perte n'est calculée que sur le contenu des tours `model` et sur le `<turn|>` qui les ferme, pour que le modèle apprenne à s'arrêter. Tout le reste est masqué (`-100`) : le prompt système, les questions et l'en-tête du tour avec son canal de pensée vide, puisque ce dernier est fourni par le template à l'inférence. Contrairement au guide officiel, qui ne démasque que le premier tour `model`, on démasque **chaque** tour, ce qui compte pour les 35 dialogues multi-tours.

In [ ]:
if RUN_TRAINING:
    END_OF_TURN_ID = tokenizer.convert_tokens_to_ids("<turn|>")


    def find_subsequence(sequence, pattern, start=0):
        for index in range(start, len(sequence) - len(pattern) + 1):
            if sequence[index : index + len(pattern)] == pattern:
                return index
        return -1


    # En-tête d'un tour du modèle (« <|turn>model\n » + préfixe de génération), tel que le tokenizer le découpe dans un vrai exemple.
    _sample_ids = processor(text=render(dataset["train"][0]["messages"]))["input_ids"][0].tolist()
    _prefix_ids = tokenizer.encode(GENERATION_PREFIX, add_special_tokens=False) if GENERATION_PREFIX else []
    _candidates = [
        tokenizer.encode(MODEL_TURN_OPEN + GENERATION_PREFIX, add_special_tokens=False),
        [tokenizer.convert_tokens_to_ids(token) for token in ("<|turn>", "model", "\n")] + _prefix_ids,
    ]
    MODEL_HEADER_IDS = next((c for c in _candidates if find_subsequence(_sample_ids, c) >= 0), None)
    assert MODEL_HEADER_IDS is not None, "En-tête du tour « model » introuvable : vérifiez le rendu du template ci-dessus."


    def model_turn_labels(input_ids):
        # Labels = input_ids sur les tours du modèle (fermeture <turn|> incluse), -100 partout ailleurs.
        labels = [-100] * len(input_ids)
        position = find_subsequence(input_ids, MODEL_HEADER_IDS)
        while position >= 0:
            index = position + len(MODEL_HEADER_IDS)
            while index < len(input_ids) and input_ids[index] != END_OF_TURN_ID:
                labels[index] = input_ids[index]
                index += 1
            if index < len(input_ids):
                labels[index] = input_ids[index]
            position = find_subsequence(input_ids, MODEL_HEADER_IDS, index)
        return labels


    def collate_fn(examples):
        texts = [render(example["messages"]) for example in examples]
        batch = processor(text=texts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LENGTH)
        labels = torch.tensor([model_turn_labels(ids) for ids in batch["input_ids"].tolist()])
        labels[batch["attention_mask"] == 0] = -100
        batch["labels"] = labels
        return batch


    # Contrôle visuel sur un dialogue multi-tours : en vert ce qui est appris, en gris ce qui est masqué.
    multi_turn = next(ex for ex in dataset["train"] if len(ex["messages"]) > 3)
    check = collate_fn([multi_turn])
    pieces = []
    for token_id, label in zip(check["input_ids"][0].tolist(), check["labels"][0].tolist()):
        piece = tokenizer.decode([token_id])
        pieces.append(f"\033[92m{piece}\033[0m" if label != -100 else f"\033[90m{piece}\033[0m")
    print("".join(pieces))
    print(f"\nTokens appris : {(check['labels'] != -100).sum().item()} / {check['labels'].numel()}")

## 10. Configuration LoRA et de l'entraînement

Sans `target_modules`, PEFT applique ses valeurs par défaut pour Gemma 4, limitées aux couches du modèle de langage. Les encodeurs vision et audio restent gelés.

In [ ]:
if RUN_TRAINING:
    from peft import LoraConfig, prepare_model_for_kbit_training
    from trl import SFTConfig, SFTTrainer

    # Recommandé pour QLoRA ; le guide officiel ne le saute que sur les GPU de 16 Go par manque de mémoire.
    if GPU_MEMORY_GB > 16:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

    peft_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM",
    )

    per_device_batch_size = 2 if GPU_MEMORY_GB >= 40 else 1

    args = SFTConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=per_device_batch_size,
        per_device_eval_batch_size=per_device_batch_size,
        gradient_accumulation_steps=max(1, EFFECTIVE_BATCH_SIZE // per_device_batch_size),
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        optim="adamw_torch_fused",
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_steps=10,
        max_grad_norm=0.3,
        bf16=torch_dtype == torch.bfloat16,
        fp16=torch_dtype == torch.float16,
        logging_steps=5,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        report_to="none",
        seed=42,
        dataset_kwargs={"skip_prepare_dataset": True},  # le collator s'occupe du rendu et de la tokenisation
        remove_unused_columns=False,
    )

    model.config.use_cache = False  # incompatible avec le gradient checkpointing
    trainer = SFTTrainer(
        model=model,
        args=args,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
        peft_config=peft_config,
        processing_class=processor,
        data_collator=collate_fn,
    )
    trainer.model.print_trainable_parameters()

## 11. Entraînement

In [ ]:
if RUN_TRAINING:
    train_result = trainer.train()
    print(train_result.metrics)

In [ ]:
if RUN_TRAINING:
    import matplotlib.pyplot as plt

    history = trainer.state.log_history
    train_points = [(h["step"], h["loss"]) for h in history if "loss" in h]
    eval_points = [(h["step"], h["eval_loss"]) for h in history if "eval_loss" in h]

    plt.figure(figsize=(8, 4))
    plt.plot(*zip(*train_points), label="perte d'entraînement")
    plt.plot(*zip(*eval_points), "o-", label="perte de validation")
    plt.xlabel("pas")
    plt.ylabel("perte")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

    print(f"Meilleur checkpoint : {trainer.state.best_model_checkpoint} (eval_loss {trainer.state.best_metric:.3f})")

Si la perte de validation remonte nettement dès la 2ᵉ époque alors que celle d'entraînement continue de baisser, le modèle apprend les réponses par cœur. Le meilleur checkpoint est déjà rechargé, mais pour le prochain essai vous pouvez baisser `NUM_EPOCHS` à 2 ou `LEARNING_RATE` à `1e-4`.

## 12. Sauvegarde de l'adaptateur

Seul l'adaptateur LoRA est sauvegardé (quelques centaines de Mo). À l'utilisation, il se recharge par-dessus `google/gemma-4-12B-it`. Le prompt système est enregistré avec lui (`system_prompt.txt`), pour que le parcours de rechargement utilise exactement le même. La copie sur Google Drive fonctionne aussi depuis VS Code (extension v0.2.1 ou plus récente).

In [ ]:
if RUN_TRAINING:
    trainer.save_model(OUTPUT_DIR)
    processor.save_pretrained(OUTPUT_DIR)
    with open(os.path.join(OUTPUT_DIR, "system_prompt.txt"), "w", encoding="utf-8") as f:
        f.write(SYSTEM_PROMPT)
    print(f"Adaptateur sauvegardé dans {OUTPUT_DIR}/")

    if SAVE_TO_DRIVE:
        import shutil

        from google.colab import drive

        drive.mount("/content/drive")
        shutil.copytree(OUTPUT_DIR, DRIVE_DIR, dirs_exist_ok=True, ignore=shutil.ignore_patterns("checkpoint-*"))
        print(f"Copié dans Google Drive : {DRIVE_DIR}")

## 13. Après l'entraînement : mêmes questions

In [ ]:
if RUN_TRAINING:
    model = trainer.model
    model.train(False)  # mode inférence : désactive le dropout LoRA
    model.config.use_cache = True

    for question in TEST_QUESTIONS:
        print(f"❓ {question}\n")
        print(f"AVANT :\n{baseline_answers[question]}\n")
        print(f"APRÈS :\n{ask(question)}\n\n{'─' * 80}\n")

Comparaison avec les réponses de référence sur des exemples de validation, jamais vus pendant l'entraînement :

In [ ]:
if RUN_TRAINING:
    import random

    for example in random.Random(0).sample(list(dataset["validation"]), 3):
        messages = example["messages"]
        question, reference = messages[-2]["content"], messages[-1]["content"]
        history = messages[1:-2]  # tours précédents éventuels, sans le prompt système
        print(f"❓ {question}\n\nRÉFÉRENCE :\n{reference}\n\nMODÈLE :\n{ask(question, history)}\n\n{'─' * 80}\n")

## 14. Recharger un adaptateur existant

Cette section est exécutée uniquement lorsque `RUN_TRAINING = False`. `ADAPTER_PATH` vaut par défaut `/content/drive/MyDrive/mitterrand-qlora`. Placer les fichiers de l’adaptateur directement dans le dossier `mitterrand-qlora` de Mon Drive, puis exécuter les cellules. Cette section monte Google Drive automatiquement. Un autre dossier local ou un identifiant Hugging Face restent possibles. L'adaptateur est appliqué au modèle de base chargé en section 4. Après un entraînement, le modèle déjà en mémoire est utilisé directement.

Le prompt sauvegardé avec l'adaptateur est prioritaire. Pour un ancien adaptateur sans prompt, on reprend `data/system_prompt.txt` du dépôt, comme dans l'ancien notebook de discussion.

In [ ]:
if not RUN_TRAINING:
    import os

    if ADAPTER_PATH.startswith("/content/drive"):
        from google.colab import drive

        drive.mount("/content/drive")

    adapter_path = Path(ADAPTER_PATH).expanduser()
    local_adapter = adapter_path if adapter_path.is_absolute() else REPO_DIR / adapter_path
    if local_adapter.is_dir():
        adapter_dir = str(local_adapter.resolve())
    elif adapter_path.is_absolute() or ADAPTER_PATH == "adapter" or ADAPTER_PATH.startswith(("./", "../")):
        raise FileNotFoundError(
            f"Adaptateur local introuvable : {local_adapter}. "
            "Placez les fichiers de l’adaptateur dans ce dossier Drive, ou corrigez ADAPTER_PATH."
        )
    else:
        from huggingface_hub import snapshot_download

        adapter_dir = snapshot_download(ADAPTER_PATH)

    required_files = ("adapter_config.json", "adapter_model.safetensors")
    missing = [name for name in required_files if not (Path(adapter_dir) / name).is_file()]
    assert not missing, f"Adaptateur incomplet dans {adapter_dir} : {missing}"

    import json

    adapter_config = json.loads((Path(adapter_dir) / "adapter_config.json").read_text(encoding="utf-8"))
    assert adapter_config["base_model_name_or_path"] == MODEL_ID, (
        f"Cet adaptateur attend {adapter_config['base_model_name_or_path']}, mais MODEL_ID vaut {MODEL_ID}."
    )

    # Copie de data/system_prompt.txt, le prompt utilisé à l'entraînement. Sert pour les adaptateurs entraînés avec
    # une version du notebook qui n'enregistrait pas encore system_prompt.txt à côté de l'adaptateur.
    DEFAULT_SYSTEM_PROMPT = (REPO_DIR / "data" / "system_prompt.txt").read_text(encoding="utf-8").strip()

    system_prompt_file = os.path.join(adapter_dir, "system_prompt.txt")
    if os.path.exists(system_prompt_file):
        with open(system_prompt_file, encoding="utf-8") as f:
            SYSTEM_PROMPT = f.read().strip()
        source = "system_prompt.txt de l'adaptateur"
    else:
        SYSTEM_PROMPT = DEFAULT_SYSTEM_PROMPT
        source = "prompt du projet (l'adaptateur n'en contient pas)"
    print(f"Adaptateur : {adapter_dir}\nFichiers : {sorted(os.listdir(adapter_dir))}\n\nPrompt système ({source}) :\n{SYSTEM_PROMPT}")

In [ ]:
if not RUN_TRAINING:
    from peft import PeftModel

    if not isinstance(model, PeftModel):  # cellule relancée : l'adaptateur est déjà en place
        model = PeftModel.from_pretrained(model, adapter_dir)  # chargé en mode inférence
    # Utiliser le tokenizer et le chat template sauvegardés lors de l'entraînement.
    processor_source = adapter_dir if all(
        (Path(adapter_dir) / name).is_file() for name in ("processor_config.json", "tokenizer.json", "chat_template.jinja")
    ) else MODEL_ID
    processor = AutoProcessor.from_pretrained(processor_source)
    print(f"Adaptateur appliqué au modèle de base. Mémoire GPU utilisée : {torch.cuda.memory_allocated() / 1024**3:.1f} Go")

### Une question rapide


In [ ]:
print(ask("Monsieur Mitterrand, que représente pour vous la date du 10 mai 1981 ?"))

## 15. Évaluation avant/après adaptation

On compare la **baseline**, soit le modèle de base avec la meilleure configuration de prompt de la section 6 (adaptateur désactivé par `model.disable_adapter()`), et le **modèle adapté**, avec le prompt système de l'entraînement. Aucune requête n'a servi à l'entraînement :

- les 30 exemples de `data/mitterrand_test.jsonl`, avec leur réponse de référence ;
- les 5 questions de la section 6 ;
- 10 requêtes écrites pour cette évaluation : 5 dialogues dont la dernière question n'a de sens qu'avec l'historique, et 5 demandes hors périmètre.

Les deux modèles reçoivent le même historique et la même graine d'échantillonnage.

**Grille, fixée avant de lire les réponses.** Chaque critère est noté de 0 à 2 :

| Critère | 0 | 1 | 2 | Cas notés |
|---|---|---|---|---|
| Pertinence et correction | hors sujet ou faux | répond en partie, ou de façon imprécise | répond à la question, en accord avec le programme de 1981 | tous |
| Rôle et format | sort du personnage, tutoie, écrit en listes ou en anglais | personnage tenu, avec un écart de registre ou de format | première personne, vouvoiement, prose en français soutenu | tous |
| Historique | ignore les tours précédents | les utilise en partie | s'appuie correctement sur les tours précédents | dialogues à plusieurs tours |
| Hors périmètre | répond comme s'il savait, ou rend le service demandé | signale la limite, mais improvise | reconnaît ne pas savoir ou décline, puis redirige | demandes hors périmètre |
| Absence d'invention | au moins un fait faux ou invérifiable important | un détail douteux mineur | aucune information inventée repérée | tous |

Un critère qui ne s'applique pas à un cas est marqué `NA` et exclu des moyennes. Deux mesures automatiques complètent la notation : la perte sur les réponses de référence du test et des indicateurs de format.

In [ ]:
import unicodedata

# Demandes hors périmètre du jeu de test et de la section 6 : sujets postérieurs au printemps 1981.
OUT_OF_SCOPE_QUESTIONS = {
    "tu penses quoi de TikTok ?",
    "Il y a eu une pandémie mondiale, le Covid. Qu'est-ce que vous auriez fait ?",
}


def turns(*pairs):
    return [{"role": role, "content": content} for role, content in pairs]


# Requêtes écrites pour l'évaluation. Dans les dialogues, la dernière question n'a de sens qu'avec l'historique.
EXTRA_CASES = [
    {"id": "hist-1", "history": turns(
        ("user", "On peut se tutoyer ?"),
        ("assistant", "Permettez-moi de garder le vouvoiement. Ce n'est pas de la distance, c'est une forme de respect."),
    ), "question": "Bon, d'accord. Alors, tu ferais quoi pour les jeunes ?"},
    {"id": "hist-2", "history": turns(
        ("user", "Que pensez-vous de Valéry Giscard d'Estaing ?"),
        ("assistant", "C'est un adversaire que je respecte. Mais je combats sa politique, qui a laissé le chômage s'installer."),
    ), "question": "Et que lui reprochez-vous le plus ?"},
    {"id": "hist-3", "history": turns(
        ("user", "Je suis libraire à Clermont-Ferrand."),
        ("assistant", "Je vous écoute."),
    ), "question": "Qu'est-ce que votre programme changera pour mon métier ?"},
    {"id": "hist-4", "history": turns(
        ("user", "Quelles sont vos deux priorités si vous êtes élu ?"),
        ("assistant", "L'emploi d'abord, car le chômage est la plaie de notre société. Ensuite, la justice sociale."),
    ), "question": "Développez la deuxième."},
    {"id": "hist-5", "history": turns(
        ("user", "J'habite à Nevers."),
        ("assistant", "La Nièvre ! C'est ma terre d'élection depuis plus de trente ans."),
    ), "question": "Qu'est-ce que la décentralisation changera pour notre département ?"},
    {"id": "hors-1", "question": "Pouvez-vous me donner une recette de gâteau au chocolat ?", "hors_perimetre": True},
    {"id": "hors-2", "question": "Résolvez l'équation 2x + 3 = 11.", "hors_perimetre": True},
    {"id": "hors-3", "question": "J'ai de la fièvre depuis trois jours, quel médicament dois-je prendre ?", "hors_perimetre": True},
    {"id": "hors-4", "question": "Qui a gagné l'élection présidentielle de 1995 ?", "hors_perimetre": True},
    {"id": "hors-5", "question": "Que pensez-vous de l'euro ?", "hors_perimetre": True},
]

categories = {}
for example in read_jsonl("mitterrand_full_with_meta.jsonl"):
    categories[example["messages"][-2]["content"]] = example["category"]

EVAL_CASES = []
for index, example in enumerate(read_jsonl("mitterrand_test.jsonl")):
    history, question, reference = split_example(example)
    EVAL_CASES.append({"id": f"test-{index:02d}", "categorie": categories.get(question, "?"), "history": history, "question": question, "reference": reference})
for index, question in enumerate(TEST_QUESTIONS, 1):
    EVAL_CASES.append({"id": f"q-{index}", "categorie": "questions de la section 6", "history": [], "question": question, "reference": ""})
for case in EXTRA_CASES:
    EVAL_CASES.append({"categorie": "hors périmètre" if case.get("hors_perimetre") else "historique", "history": [], "reference": "", **case})
for case in EVAL_CASES:
    case["hors_perimetre"] = case.get("hors_perimetre", False) or case["question"] in OUT_OF_SCOPE_QUESTIONS


def normalize(text):
    text = unicodedata.normalize("NFKD", text.lower())
    return re.sub(r"[^a-z0-9]+", " ", "".join(char for char in text if not unicodedata.combining(char))).strip()


seen = {normalize(message["content"]) for example in train_examples + val_examples for message in example["messages"] if message["role"] == "user"}
leaks = [case["id"] for case in EVAL_CASES if normalize(case["question"]) in seen]
assert not leaks, f"Requêtes déjà présentes dans l'entraînement ou la validation : {leaks}"
print(f"{len(EVAL_CASES)} cas, aucun vu à l'entraînement ou en validation.")
print(f"Avec historique : {sum(bool(case['history']) for case in EVAL_CASES)} · hors périmètre : {sum(case['hors_perimetre'] for case in EVAL_CASES)}")

### Génération des réponses

Les réponses sont enregistrées au fil de l'eau dans `RESULTS_DIR/reponses.json` : une session interrompue reprend là où elle s'était arrêtée, et une session suivante relit les réponses au lieu de les régénérer. `REGENERATE_EVAL = True` force une nouvelle génération.

In [ ]:
import time

REGENERATE_EVAL = False  # @param {type:"boolean"}
EVAL_SEED = 42
answers_file = RESULTS_DIR / "reponses.json"

eval_answers = {}
if answers_file.is_file() and not REGENERATE_EVAL:
    eval_answers = json.loads(answers_file.read_text(encoding="utf-8"))

model.train(False)
start = time.time()
for number, case in enumerate(EVAL_CASES, 1):
    saved = eval_answers.get(case["id"])
    if saved and saved["question"] == case["question"]:
        continue
    set_seed(EVAL_SEED)
    with model.disable_adapter():  # baseline : même modèle, adaptateur désactivé, meilleure configuration de prompt
        baseline = ask(case["question"], case["history"], **BASELINE_CONFIG)
    set_seed(EVAL_SEED)
    adapted = ask(case["question"], case["history"])
    eval_answers[case["id"]] = {"question": case["question"], "baseline": baseline, "adapte": adapted}
    answers_file.write_text(json.dumps(eval_answers, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"{number}/{len(EVAL_CASES)} {case['id']} ({time.time() - start:.0f} s)")
print(f"Réponses : {answers_file}")

### Mesures automatiques

La perte est calculée sur les 30 réponses de référence du jeu de test, avec le prompt de chaque modèle. Les indicateurs de format portent sur toutes les réponses générées.

In [ ]:
test_cases = [case for case in EVAL_CASES if case["reference"]]
test_losses = {"baseline": [], "adapte": []}
for case in test_cases:
    with model.disable_adapter():
        test_losses["baseline"].append(reference_loss(build_messages(case["question"], case["history"], **BASELINE_CONFIG), case["reference"]))
    test_losses["adapte"].append(reference_loss(build_messages(case["question"], case["history"]), case["reference"]))

automatic = {}
for model_name, label in (("baseline", "baseline"), ("adapte", "adapté")):
    indicators = [format_indicators(eval_answers[case["id"]][model_name]) for case in EVAL_CASES]
    mean_loss = sum(test_losses[model_name]) / len(test_losses[model_name])
    automatic[label] = {
        f"perte sur les références du test ({len(test_cases)})": round(mean_loss, 3),
        "perplexité": round(math.exp(mean_loss), 1),
        "réponses avec listes ou markdown": f"{sum(i['markdown'] for i in indicators)}/{len(indicators)}",
        "réponses avec tutoiement": f"{sum(i['tutoiement'] for i in indicators)}/{len(indicators)}",
        "longueur moyenne (mots)": round(sum(i["mots"] for i in indicators) / len(indicators)),
    }
display(pd.DataFrame(automatic))

### Notation humaine

La cellule suivante crée `grille.csv` dans `RESULTS_DIR`, avec une ligne par cas et par modèle : l'historique, la question, la référence, la réponse, puis une colonne par critère (`NA` quand il ne s'applique pas). Pour noter :

1. télécharger `grille.csv` (depuis Drive ou le panneau Fichiers de Colab) et l'ouvrir dans un tableur ;
2. remplir chaque critère avec 0, 1 ou 2 selon la grille ci-dessus, et la colonne `commentaire` pour les cas notables ;
3. l'enregistrer au format CSV au même emplacement, puis exécuter les cellules suivantes.

La cellule ne remplace jamais une grille existante.

In [ ]:
CRITERIA = {
    "pertinence": "Pertinence et correction",
    "role_format": "Rôle et format",
    "historique": "Historique",
    "hors_perimetre": "Hors périmètre",
    "absence_invention": "Absence d'invention",
}
scores_file = RESULTS_DIR / "grille.csv"

if scores_file.is_file():
    print(f"Grille existante conservée : {scores_file}")
else:
    rows = []
    for case in EVAL_CASES:
        for model_name in ("baseline", "adapte"):
            rows.append({
                "id": case["id"],
                "modele": model_name,
                "categorie": case["categorie"],
                "historique_du_dialogue": "\n".join(f"{message['role']} : {message['content']}" for message in case["history"]),
                "question": case["question"],
                "reference": case["reference"],
                "reponse": eval_answers[case["id"]][model_name],
                "pertinence": "",
                "role_format": "",
                "historique": "" if case["history"] else "NA",
                "hors_perimetre": "" if case["hors_perimetre"] else "NA",
                "absence_invention": "",
                "commentaire": "",
            })
    pd.DataFrame(rows).to_csv(scores_file, index=False, encoding="utf-8-sig")
    print(f"Grille à remplir : {scores_file} ({len(rows)} lignes)")

In [ ]:
scores = pd.read_csv(scores_file, encoding="utf-8-sig", dtype=str, keep_default_na=False)
for column in CRITERIA:
    scores[column] = scores[column].str.strip()
invalid = ~scores[list(CRITERIA)].isin(["0", "1", "2", "NA", ""]).all(axis=1)
assert not invalid.any(), f"Notes invalides (0, 1, 2 ou NA attendus) aux lignes {[index + 2 for index in scores.index[invalid]]} de {scores_file}"

notes = scores.melt(id_vars=["id", "modele"], value_vars=list(CRITERIA), var_name="critere", value_name="note")
to_score = int((notes["note"] == "").sum())
notes = notes[notes["note"].isin(["0", "1", "2"])].astype({"note": int})

# Comparaison appariée : un cas ne compte que s'il est noté pour les deux modèles.
paired = notes.pivot_table(index=["critere", "id"], columns="modele", values="note")
paired = paired.dropna() if {"baseline", "adapte"} <= set(paired.columns) else paired.iloc[0:0]

if to_score:
    print(f"{to_score} notes restent à saisir dans {scores_file}.")
if paired.empty:
    print("Aucune note appariée pour l'instant : remplir la grille, puis relancer cette cellule.")
else:
    summary = paired.groupby(level="critere").agg(["mean", "count"])
    table = pd.DataFrame({
        "notes par modèle": summary[("adapte", "count")].astype(int),
        "baseline (0-2)": summary[("baseline", "mean")].round(2),
        "adapté (0-2)": summary[("adapte", "mean")].round(2),
    }).reindex([criterion for criterion in CRITERIA if criterion in summary.index])
    table.loc["ensemble"] = [len(paired), round(paired["baseline"].mean(), 2), round(paired["adapte"].mean(), 2)]
    table["notes par modèle"] = table["notes par modèle"].astype(int)
    table["écart"] = (table["adapté (0-2)"] - table["baseline (0-2)"]).round(2)
    table.index = [CRITERIA.get(criterion, "Ensemble des notes") for criterion in table.index]
    display(table)

In [ ]:
def show_case(case_id):
    # Affiche un cas avec les deux réponses et leurs notes, pour commenter réussites et erreurs.
    case = next(case for case in EVAL_CASES if case["id"] == case_id)
    print(f"[{case_id}] {case['categorie']}\n")
    for message in case["history"]:
        print(f"{message['role']} : {message['content']}")
    print(f"❓ {case['question']}\n")
    if case["reference"]:
        print(f"RÉFÉRENCE :\n{case['reference']}\n")
    for model_name, label in (("baseline", "BASELINE"), ("adapte", "ADAPTÉ")):
        print(f"{label} :\n{eval_answers[case_id][model_name]}")
        row = scores[(scores["id"] == case_id) & (scores["modele"] == model_name)]
        if not row.empty:
            given = ", ".join(f"{CRITERIA[c]} {row.iloc[0][c]}" for c in CRITERIA if row.iloc[0][c] not in ("", "NA"))
            print(f"→ {given or 'non noté'}  {row.iloc[0]['commentaire']}")
        print()


show_case("hist-1")

### Réussites et erreurs commentées

*À rédiger après la notation, avec `show_case("<id>")` : au moins cinq cas, réussites et erreurs, en citant leur identifiant et le critère concerné.*

1. …
2. …
3. …
4. …
5. …

## 16. Interface Gradio

L'app de `app/app.py` appelle directement `ask()`, dans le même environnement que le modèle. Elle conserve son style, ses portraits, ses suggestions, son historique de cinq échanges et sa réinitialisation.

Dans Colab, l'interface s'affiche sous la cellule via le proxy interne (`share=False`). Utiliser Chrome ou Firefox. Depuis VS Code avec un serveur Colab, ouvrir le notebook dans Colab pour afficher Gradio.

In [ ]:
import inspect
from app.app import launch_app

assert "respond" in inspect.signature(launch_app).parameters, (
    f"L'app de {REPO_DIR} ne sait pas encore appeler le modèle : choisissez la branche qui contient launch_app(respond=...)."
)
launch_app(respond=ask)
